# <span style="color:rgb(213,80,0)">ROZWIĄZYWANIE RÓWNAŃ NIELINIOWYCH</span>

## PARAMETRY LOKALNEJ ZBIEŻNOŚCI

Teraz zajmijmy się drugim istotnym parametrem zbieżności — wykładnikiem zbieżności $\varrho$ .


$$\Delta_{i+1} =C\Delta_i^{\varrho }$$

Rozwiążemy nasz problem za pomocą dwóch modyfikacji metody Newtona: samą metodą Newtona i metodą *regula falsi*.

In [ ]:
import numpy as np
import warnings

def calculateNewtonSolution(f, df, x0, opts=None):
    if opts is None:
        opts = {}
    imax = opts.get('imax', 1000000)
    return_all = opts.get('return_all', False)
    TOL = opts.get('TOL', 1e-12)
    show_warnings = opts.get('warnings', True)
    
    if return_all:
        x = np.zeros(imax)
        i = 0
        x[0] = x0 - f(x0)/df(x0)
        err = np.abs(x[0] - x0)
        while err >= TOL and i < imax - 1:
            i += 1
            x[i] = x[i-1] - f(x[i-1])/df(x[i-1])
            err = np.abs(x[i] - x[i-1])
        if show_warnings and i >= imax - 1:
            warnings.warn(f'Metoda nie zbiegła po {imax} iteracjach.')
        return x[:i+1]
    else:
        x_new = x0 - f(x0)/df(x0)
        err = np.abs(x_new - x0)
        i = 0
        while err >= TOL and i < imax:
            x_old = x_new
            x_new = x_old - f(x_old)/df(x_old)
            err = np.abs(x_new - x_old)
            i += 1
        return x_new

def calculateFalsePositionSolution(f, a, b, opts=None):
    if opts is None:
        opts = {}
    imax = opts.get('imax', 1000000)
    return_all = opts.get('return_all', False)
    TOL = opts.get('TOL', 1e-12)
    show_warnings = opts.get('warnings', True)
    
    if return_all:
        x = np.zeros(imax)
        fx1, fx2 = f(a), f(b)
        if fx1 * fx2 >= 0:
            raise ValueError('Błędny wybór nawiasu.')
        c = b - (fx2 * (b - a)) / (fx2 - fx1)
        x[0] = c
        fc = f(c)
        x1, x2, fx1, fx2 = a, b, fx1, fx2
        i = 0; c_old = 0
        while abs(c - c_old) > TOL and i < imax - 1:
            if fc < 0:
                x1, fx1 = c, fc
            else:
                x2, fx2 = c, fc
            c_old = c
            c = x1 - (fx1 * (x1 - x2)) / (fx1 - fx2)
            i += 1
            x[i] = c
            fc = f(c)
        return x[:i+1]
    else:
        fx1, fx2 = f(a), f(b)
        if fx1 * fx2 >= 0:
            raise ValueError('Błędny wybór nawiasu.')
        c = b - (fx2 * (b - a)) / (fx2 - fx1)
        fc = f(c)
        x1, x2, fx1, fx2 = a, b, fx1, fx2
        i = 0; c_old = 0
        while abs(c - c_old) > TOL and i < imax:
            if fc < 0:
                x1, fx1 = c, fc
            else:
                x2, fx2 = c, fc
            c_old = c
            c = x1 - (fx1 * (x1 - x2)) / (fx1 - fx2)
            fc = f(c)
            i += 1
        return c

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

f = lambda x: (x-1)*(x-2)*(x-5)
df = lambda x: (x-1)*(x-2) + (x-1)*(x-5) + (x-2)*(x-5)
true_root = 1

options = {'imax': 100, 'return_all': True, 'TOL': 1e-8, 'warnings': True}
a, b = 0, 1.5
x0 = -1

resultsNewton = calculateNewtonSolution(f, df, x0, options)
errorNewton = np.abs(resultsNewton - true_root)

resultsFP = calculateFalsePositionSolution(f, a, b, options)
errorFP = np.abs(resultsFP - true_root)

plt.plot(errorNewton, 'ro', label='Newton')
plt.plot(errorFP, 'bo', label='Regula falsi')
plt.xlabel('Iteracja')
plt.ylabel('Błąd')
plt.grid(True)
plt.legend()
plt.show()


Te dwie metody różnią się wykładnikiem zbieżności. Zauważ ile iteracji zrobiła metoda Newtona a ile metoda *regula falsi*. Aby to lepiej pokazać wykreślmy błąd rozwiązania w funkcji numeru iteracji w skali półlogarytmicznej:

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

plt.semilogy(errorNewton, 'ro', label='Newton')
plt.semilogy(errorFP, 'bo', label='Regula falsi')
plt.xlabel('Iteracja')
plt.ylabel('Błąd')
plt.grid(True)
plt.legend()
plt.show()


Widać wyraźnie, że wykładnik lokalnej zbieżności jest znacznie silniejszy od wpływu współczynnika. Radykalnie zmienia ilość iteracji potrzebnych do osiągnięcia rozwiązania. Jest to jednak okupione najczęściej zmniejszeniem zakresu w jakim dana metoda daje wynik (najczęściej chodzi o zakres punktów startowych, rozwiązań do których może być zbieżna lub funkcji dla których działa)


**ĆWICZENIE 1:**


Spróbuj znaleźć taki zestaw danych wejściowych dla którego metoda Newtona będzie rozbieżna albo będzie wymagała zbliżonej ilości iteracji jak metoda *regula falsi*. Czy potrafisz wyjaśnić dlaczego może się tak zdarzyć mimo, że ma ona większy wykładnik zbieżności?


Podpowiedź: Kluczowe jest tutaj położenie punktu startowego


Efekt który obserwowaliśmy powyżej może zostać doprowadzony do przypadku skrajnego. Przykładowo w poniższym kodzie:

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

f2 = lambda x: x**3 - 2*x + 2
df2 = lambda x: 3*x**2 - 2

options = {'imax': 100, 'return_all': True, 'TOL': 1e-8, 'warnings': True}
x0 = 1

resultsNewton = calculateNewtonSolution(f2, df2, x0, options)
true_root2 = -1.7692923542386
errorNewton = np.abs(resultsNewton - true_root2)

plt.plot(errorNewton, 'ro')
plt.xlabel('Iteracja')
plt.ylabel('Błąd')
plt.grid(True)
plt.show()


To klasyczny przykład cyklu. Zwróć uwagę, że to czy metoda wpadnie w cykl zależy od położenia punktu startowego.


**ĆWICZENIE 2:**


Zmodyfikuj powyższy kod tak, aby metoda stała się zbieżna.


**ĆWICZENIE 3:**


Korzystając z definicji parametrów zbieżności:


$$\Delta_{i+1} =C\Delta_i^{\varrho }$$

Zmodyfikuj poniższy kod tak, żeby pokazać kwadratową zależność błędu w metodzie Newtona ($\varrho =2$)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

f2 = lambda x: x**3 - 2*x + 2
df2 = lambda x: 3*x**2 - 2

options = {'imax': 100, 'return_all': True, 'TOL': 1e-8, 'warnings': True}
x0 = 1

resultsNewton = calculateNewtonSolution(f2, df2, x0, options)

# Uzupełnij poniżej — pokaż kwadratową zależność błędu (\varrho=2)


| Metoda | Wykładnik zbieżności $\varrho$ | Współczynnik $C$ | Zakres zbieżności |
|--------|-------------------------------|-------------------|--------------------|
| Bisekcja | 1 | 0.5 | Bardzo szeroki |
| Regula falsi | 1 | Zależny od funkcji | Szeroki |
| Secant | ~1.618 | Zależny od funkcji | Umiarkowany |
| Newton | 2 | $f''(x^*)/(2f'(x^*))$ | Zależy od punktu startowego |
| Newton modyfikowany | 2 | Zmniejszony | Poszerzony |